In [1]:
# note to self: record the environment first so the results table can say what machine produced it
import platform, sys, psutil

!java -version 2>&1 | head -n 2
!pip -q install pyspark pyarrow

import pyspark
print("python", sys.version.split()[0], "| pyspark", pyspark.__version__)
print("cpus", psutil.cpu_count(), "| ram GB", round(psutil.virtual_memory().total / 1e9, 1))
print(platform.platform())

[0.018s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.018s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
python 3.13.18 | pyspark 4.0.5
cpus 2 | ram GB 13.6
Linux-6.6.159+-x86_64-with-glibc2.39


In [2]:
%%time
# note to self: downloading from uci inside colab avoids uploading 2.8 GB from my laptop
!mkdir -p /content/data
!wget -q -O /content/data/HIGGS.csv.gz https://archive.ics.uci.edu/ml/machine-learning-databases/00280/HIGGS.csv.gz
!gunzip -kf /content/data/HIGGS.csv.gz
!ls -lh /content/data
!df -h /content | tail -n 1

total 11G
-rw-r--r-- 1 root root 7.5G Nov  9 09:33 HIGGS.csv
-rw-r--r-- 1 root root 2.7G Nov  9 09:33 HIGGS.csv.gz
overlay         108G   11G   98G  10% /
CPU times: user 684 ms, sys: 83.4 ms, total: 767 ms
Wall time: 5min 9s


In [3]:
# note to self: spark 4 needs java 17 or newer, so I check the real version line this time
!java -version 2>&1 | grep -i "version"

openjdk version "21.0.12.1" 2026-08-18


In [4]:
%%time
# note to self: head on the same csv gives both engines the exact same rows (first N rows),
# and it means csv vs parquet comparisons later use identical data
!head -n 1000000 /content/data/HIGGS.csv > /content/data/higgs_1m.csv
!head -n 5000000 /content/data/HIGGS.csv > /content/data/higgs_5m.csv
!ln -sf /content/data/HIGGS.csv /content/data/higgs_11m.csv
!wc -l /content/data/higgs_1m.csv /content/data/higgs_5m.csv /content/data/HIGGS.csv
!ls -lh /content/data/*.csv

    1000000 /content/data/higgs_1m.csv
    5000000 /content/data/higgs_5m.csv
   11000000 /content/data/HIGGS.csv
   17000000 total
lrwxrwxrwx 1 root root   23 Nov  9 09:36 /content/data/higgs_11m.csv -> /content/data/HIGGS.csv
-rw-r--r-- 1 root root 697M Nov  9 09:36 /content/data/higgs_1m.csv
-rw-r--r-- 1 root root 3.5G Nov  9 09:36 /content/data/higgs_5m.csv
-rw-r--r-- 1 root root 7.5G Nov  9 09:33 /content/data/HIGGS.csv
CPU times: user 210 ms, sys: 28.3 ms, total: 239 ms
Wall time: 1min 18s


In [5]:
%%writefile /content/pandas_pipeline.py
import json
import sys
import time

import pandas as pd

# note to self: same 29 columns as my local parquet, label first then 21 low-level and 7 high-level features
COLS = ["label",
        "lepton_pt", "lepton_eta", "lepton_phi", "missing_energy_magnitude", "missing_energy_phi",
        "jet1_pt", "jet1_eta", "jet1_phi", "jet1_btag", "jet2_pt", "jet2_eta", "jet2_phi", "jet2_btag",
        "jet3_pt", "jet3_eta", "jet3_phi", "jet3_btag", "jet4_pt", "jet4_eta", "jet4_phi", "jet4_btag",
        "m_jj", "m_jjj", "m_lv", "m_jlv", "m_bb", "m_wbb", "m_wwbb"]
FEATS = COLS[1:]


def main(src, dst):
    t0 = time.perf_counter()
    # note to self: pipeline order is read, drop nulls, standardize, cast to float32, write parquet.
    # the spark script uses the exact same order so the comparison is fair
    df = pd.read_csv(src, header=None, names=COLS)  # float64 by default
    t_read = time.perf_counter()
    df = df.dropna()
    t_clean = time.perf_counter()
    # note to self: std() here is the sample std (ddof=1), same as spark's stddev
    df[FEATS] = (df[FEATS] - df[FEATS].mean()) / df[FEATS].std()
    t_scale = time.perf_counter()
    df = df.astype("float32")
    df.to_parquet(dst, index=False)
    t_end = time.perf_counter()

    print(json.dumps({
        "rows": int(len(df)), "startup_s": 0.0, "pipeline_s": t_end - t0,
        "read_s": t_read - t0, "clean_s": t_clean - t_read,
        "scale_s": t_scale - t_clean, "write_s": t_end - t_scale,
    }))


if __name__ == "__main__":
    main(sys.argv[1], sys.argv[2])

Writing /content/pandas_pipeline.py


In [6]:
%%writefile /content/spark_pipeline.py
import argparse
import json
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StructField, StructType

COLS = ["label",
        "lepton_pt", "lepton_eta", "lepton_phi", "missing_energy_magnitude", "missing_energy_phi",
        "jet1_pt", "jet1_eta", "jet1_phi", "jet1_btag", "jet2_pt", "jet2_eta", "jet2_phi", "jet2_btag",
        "jet3_pt", "jet3_eta", "jet3_phi", "jet3_btag", "jet4_pt", "jet4_eta", "jet4_phi", "jet4_btag",
        "m_jj", "m_jjj", "m_lv", "m_jlv", "m_bb", "m_wbb", "m_wwbb"]
FEATS = COLS[1:]

ap = argparse.ArgumentParser()
ap.add_argument("src")
ap.add_argument("dst")
ap.add_argument("--partitions", type=int, default=0)
ap.add_argument("--driver-memory", default="8g")
args = ap.parse_args()

t0 = time.perf_counter()
spark = (SparkSession.builder.master("local[*]").appName("higgs")
         .config("spark.driver.memory", args.driver_memory)
         .config("spark.ui.enabled", "false")  # note to self: no web ui, less overhead
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
t_start = time.perf_counter()

# note to self: explicit schema so spark does not need an extra pass over the csv to infer types
schema = StructType([StructField(c, DoubleType(), True) for c in COLS])
df = spark.read.csv(args.src, schema=schema, header=False).dropna()

# note to self: stddev is the sample std, same as pandas. this agg is one full pass over the csv,
# and the write below is a second pass. that is the natural spark way, and I will say so in the report
stats = df.agg(*[F.mean(c).alias(f"m_{c}") for c in FEATS],
               *[F.stddev(c).alias(f"s_{c}") for c in FEATS]).first()
df = df.select(
    F.col("label").cast("float"),
    *[((F.col(c) - stats[f"m_{c}"]) / stats[f"s_{c}"]).cast("float").alias(c) for c in FEATS],
)
if args.partitions:
    df = df.repartition(args.partitions)
df.write.mode("overwrite").parquet(args.dst)
t_end = time.perf_counter()

# note to self: row count is taken after the timed part so it does not inflate the pipeline time
rows = spark.read.parquet(args.dst).count()
print(json.dumps({"rows": rows, "startup_s": t_start - t0, "pipeline_s": t_end - t_start}))
spark.stop()

Writing /content/spark_pipeline.py


In [7]:
import json
import os
import shutil
import subprocess
import time

import pandas as pd
import psutil

os.makedirs("/content/out", exist_ok=True)
os.makedirs("/content/logs", exist_ok=True)
os.makedirs("/content/results", exist_ok=True)

SIZES = {"1m": "/content/data/higgs_1m.csv", "5m": "/content/data/higgs_5m.csv",
         "11m": "/content/data/higgs_11m.csv"}
CSV_OUT = "/content/results/spark_vs_pandas.csv"


def run_one(engine, size, partitions=0):
    name = f"{engine}_{size}" + (f"_p{partitions}" if partitions else "")
    dst = f"/content/out/{name}.parquet"
    if os.path.isdir(dst):
        shutil.rmtree(dst)
    elif os.path.exists(dst):
        os.remove(dst)

    cmd = ["python", f"/content/{engine}_pipeline.py", SIZES[size], dst]
    if engine == "spark" and partitions:
        cmd += ["--partitions", str(partitions)]

    # note to self: stdout/stderr go to files, a full pipe buffer could freeze the worker while I poll it
    with open(f"/content/logs/{name}.out", "w") as fo, open(f"/content/logs/{name}.err", "w") as fe:
        proc = subprocess.Popen(cmd, stdout=fo, stderr=fe)
        p = psutil.Process(proc.pid)
        peak, t0 = 0, time.perf_counter()
        while proc.poll() is None:
            try:
                # note to self: sum the worker AND its children, spark runs the work in a separate java process
                rss = p.memory_info().rss + sum(c.memory_info().rss for c in p.children(recursive=True))
            except psutil.Error:
                rss = 0
            peak = max(peak, rss)
            time.sleep(0.2)
        wall = time.perf_counter() - t0

    row = {"engine": engine, "size": size, "partitions": partitions, "returncode": proc.returncode,
           "status": "ok" if proc.returncode == 0 else "failed",
           "wall_s": round(wall, 1), "peak_gb": round(peak / 1e9, 2)}
    if proc.returncode == 0:
        lines = open(f"/content/logs/{name}.out").read().strip().splitlines()
        row.update(json.loads(lines[-1]))
    # note to self: write after every run so a session crash cannot lose finished results
    pd.DataFrame([row]).to_csv(CSV_OUT, mode="a", header=not os.path.exists(CSV_OUT), index=False)
    return row


smoke = [run_one("pandas", "1m"), run_one("spark", "1m")]
print(pd.DataFrame(smoke).to_string(index=False))

engine size  partitions  returncode status  wall_s  peak_gb    rows  startup_s  pipeline_s    read_s  clean_s  scale_s  write_s
pandas   1m           0           0     ok    21.8     0.99 1000000   0.000000   20.385929 16.372484 0.353543 1.535193  2.12471
 spark   1m           0           0     ok   118.2     1.58 1000000  17.572422   97.418822       NaN      NaN      NaN      NaN


In [8]:
# note to self: cheap runs first, pandas at 11m last because it may run out of memory
plan = [("pandas", "5m"), ("spark", "5m"), ("spark", "11m"), ("pandas", "11m")]
results = []
for engine, size in plan:
    print(f"running {engine} {size} ...", flush=True)
    r = run_one(engine, size)
    results.append(r)
    print({k: r[k] for k in ["engine", "size", "status", "wall_s", "peak_gb"]}, flush=True)

print(pd.DataFrame(results).to_string(index=False))

running pandas 5m ...
{'engine': 'pandas', 'size': '5m', 'status': 'ok', 'wall_s': 95.2, 'peak_gb': 4.6}
running spark 5m ...
{'engine': 'spark', 'size': '5m', 'status': 'ok', 'wall_s': 361.9, 'peak_gb': 1.74}
running spark 11m ...
{'engine': 'spark', 'size': '11m', 'status': 'ok', 'wall_s': 735.4, 'peak_gb': 1.76}
running pandas 11m ...
{'engine': 'pandas', 'size': '11m', 'status': 'ok', 'wall_s': 206.1, 'peak_gb': 10.08}
engine size  partitions  returncode status  wall_s  peak_gb     rows  startup_s  pipeline_s     read_s  clean_s   scale_s   write_s
pandas   5m           0           0     ok    95.2     4.60  5000000   0.000000   91.988242  72.040164 1.994101  7.757453 10.196523
 spark   5m           0           0     ok   361.9     1.74  5000000  16.644424  341.765442        NaN      NaN       NaN       NaN
 spark  11m           0           0     ok   735.4     1.76 11000000  15.929670  714.910960        NaN      NaN       NaN       NaN
pandas  11m           0           0     ok   

In [9]:
# note to self: sweep on 1m only, it takes about 2 minutes per setting on 2 cpus
sweep = [run_one("spark", "1m", partitions=p) for p in (4, 8, 16, 32)]
print(pd.DataFrame(sweep)[["partitions", "status", "wall_s", "peak_gb", "pipeline_s"]].to_string(index=False))

 partitions status  wall_s  peak_gb  pipeline_s
          4     ok   122.3     1.38   98.375552
          8     ok   112.2     1.33   94.053177
         16     ok   134.8     1.32  115.814096
         32     ok   116.3     1.34   98.072250


In [11]:
rows = []
for size in ["1m", "5m", "11m"]:
    pq_path = f"/content/out/pandas_{size}.parquet"
    t0 = time.perf_counter()
    df = pd.read_parquet(pq_path)
    t_pq = time.perf_counter() - t0
    del df
    res = pd.read_csv(CSV_OUT)
    # note to self: csv read time comes from the earlier pandas runs (read_s), the parquet read is timed here
    csv_read = res[(res.engine == "pandas") & (res["size"] == size) & (res.partitions == 0)]["read_s"].iloc[0]
    rows.append({"size": size,
                 "csv_gb": round(os.path.getsize(os.path.realpath(SIZES[size])) / 1e9, 2),
                 "parquet_gb": round(os.path.getsize(pq_path) / 1e9, 2),
                 "read_csv_s": round(csv_read, 1), "read_parquet_s": round(t_pq, 1)})
fmt = pd.DataFrame(rows)
fmt.to_csv("/content/results/csv_vs_parquet.csv", index=False)
print(fmt.to_string(index=False))

size  csv_gb  parquet_gb  read_csv_s  read_parquet_s
  1m    0.73        0.06        16.4             1.1
  5m    3.65        0.31        72.0             3.0
 11m    8.04        0.69       157.4             6.6


In [12]:
from google.colab import files
files.download(CSV_OUT)
files.download("/content/results/csv_vs_parquet.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>